In [1]:
# ============================================
# Notebook-friendly RQ1 retrieval evaluation
# PEFT-safe + memory-safe Qwen loading
# with retrieval preview printing
# ============================================

import gc
import json
import os
from typing import Dict, List, Tuple, Any

import numpy as np
import torch
from tqdm import tqdm
from transformers import (
    AutoModel,
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import PeftConfig, PeftModel


# ---------------------------
# Config
# ---------------------------
class Args:
    data_path = "dataset/test.json"
    qwen_path = "./qwen25-coder-sparql2text"
    sentence_model = "sentence-transformers/all-MiniLM-L6-v2"
    use_field = "sparql_dbpedia18"   # or "sparql_wikidata"

    answer_jaccard_threshold = 0.8
    topk = [1, 5, 10]

    run_sparqlce = False
    run_qwen = True

    qwen_gen_batch_size = 1
    qwen_max_new_tokens = 32
    sentence_batch_size = 32

    seed = 42
    save_dir = "results_rq1_notebook"

    # set to None for full dataset after sanity check
    max_examples = None

    # retrieval preview
    preview_examples = 5


args = Args()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


# ---------------------------
# Utilities
# ---------------------------
def cleanup_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def set_seed(seed: int = 42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def l2_normalize(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.clip(norms, eps, None)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    a = l2_normalize(a)
    b = l2_normalize(b)
    return a @ b.T


def safe_list(x: Any) -> List[str]:
    if x is None:
        return []
    if isinstance(x, list):
        return [str(v).strip() for v in x if str(v).strip()]
    s = str(x).strip()
    return [s] if s else []


def normalize_answer_item(x: str) -> str:
    return " ".join(str(x).strip().split())


def normalize_answer_list(answer: Any) -> List[str]:
    vals = safe_list(answer)
    return [normalize_answer_item(v) for v in vals if normalize_answer_item(v)]


def answer_jaccard(a: List[str], b: List[str]) -> float:
    sa, sb = set(a), set(b)
    if len(sa) == 0 and len(sb) == 0:
        return 1.0
    if len(sa) == 0 or len(sb) == 0:
        return 0.0
    return len(sa.intersection(sb)) / len(sa.union(sb))


def save_json(obj: Dict, path: str):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


# ---------------------------
# Data loading
# ---------------------------
def load_json(path: str) -> List[Dict]:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def load_examples(data_path: str, use_field: str, max_examples: int = None) -> List[Dict]:
    raw = load_json(data_path)

    examples = []
    for idx, ex in enumerate(raw):
        sparql = ex.get(use_field)
        question = ex.get("question")
        answers = normalize_answer_list(ex.get("answer"))

        if not sparql or not str(sparql).strip():
            continue
        if not question or not str(question).strip():
            continue

        examples.append(
            {
                "id": ex.get("id", idx),
                "sparql": str(sparql).strip(),
                "question": str(question).strip(),
                "answer": answers,
            }
        )

        if max_examples is not None and len(examples) >= max_examples:
            break

    return examples


# ---------------------------
# Ground truth
# ---------------------------
def build_ground_truth_by_answer_jaccard(
    examples: List[Dict],
    threshold: float = 0.8,
    exclude_self: bool = True,
) -> Tuple[List[str], List[str], List[str], Dict[int, List[int]], np.ndarray]:
    n = len(examples)
    corpus_sparql = [ex["sparql"] for ex in examples]
    anchor_sparql = corpus_sparql
    corpus_questions = [ex["question"] for ex in examples]

    jaccard_matrix = np.zeros((n, n), dtype=np.float32)
    for i in tqdm(range(n), desc="Building answer-Jaccard matrix"):
        ai = examples[i]["answer"]
        for j in range(n):
            if exclude_self and i == j:
                continue
            aj = examples[j]["answer"]
            jaccard_matrix[i, j] = answer_jaccard(ai, aj)

    gt = {}
    for i in range(n):
        positives = [j for j in range(n) if jaccard_matrix[i, j] >= threshold]
        if exclude_self:
            positives = [j for j in positives if j != i]
        gt[i] = positives

    return corpus_sparql, anchor_sparql, corpus_questions, gt, jaccard_matrix


# ---------------------------
# SPARQLCE stub
# ---------------------------
def encode_sparqlce_batch(queries: List[str], batch_size: int = 16) -> np.ndarray:
    raise NotImplementedError("Plug in your SPARQLCE encoder here.")


# ---------------------------
# Sentence encoder
# ---------------------------
class SentenceEncoder:
    def __init__(self, model_name: str, device: str = DEVICE):
        self.device = device
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(device)
        self.model.eval()

    @torch.no_grad()
    def encode(self, texts: List[str], batch_size: int = 32, max_length: int = 256) -> np.ndarray:
        all_vecs = []

        for start in tqdm(range(0, len(texts), batch_size), desc="Sentence encode", leave=False):
            batch = texts[start : start + batch_size]
            toks = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=max_length,
            ).to(self.device)

            out = self.model(**toks)
            last_hidden = out.last_hidden_state
            mask = toks["attention_mask"].unsqueeze(-1).float()

            summed = (last_hidden * mask).sum(dim=1)
            counts = mask.sum(dim=1).clamp(min=1e-9)
            pooled = summed / counts

            all_vecs.append(pooled.cpu().numpy())

        return np.concatenate(all_vecs, axis=0)


# ---------------------------
# Qwen baseline
# ---------------------------
class QwenTextBaseline:
    def __init__(self, qwen_path: str, sentence_model_name: str, device: str = DEVICE):
        self.device = device
        self.qwen_path = qwen_path

        peft_cfg = PeftConfig.from_pretrained(qwen_path)
        self.base_model_name = peft_cfg.base_model_name_or_path

        try:
            self.qwen_tokenizer = AutoTokenizer.from_pretrained(qwen_path)
        except Exception:
            self.qwen_tokenizer = AutoTokenizer.from_pretrained(self.base_model_name)

        if self.qwen_tokenizer.pad_token is None:
            self.qwen_tokenizer.pad_token = self.qwen_tokenizer.eos_token

        if device == "cuda":
            bnb_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_use_double_quant=True,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_compute_dtype=torch.float16,
            )

            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_name,
                quantization_config=bnb_config,
                device_map="auto",
                low_cpu_mem_usage=True,
            )

            self.qwen_model = PeftModel.from_pretrained(
                base_model,
                qwen_path,
                is_trainable=False,
            )
        else:
            base_model = AutoModelForCausalLM.from_pretrained(
                self.base_model_name,
                dtype=torch.float32,
                low_cpu_mem_usage=True,
            ).to(device)

            self.qwen_model = PeftModel.from_pretrained(
                base_model,
                qwen_path,
                is_trainable=False,
            )

        self.qwen_model.eval()
        self.sentence_encoder = SentenceEncoder(sentence_model_name, device=device)

    def build_prompt(self, sparql: str) -> str:
        return (
            "You are an expert in knowledge graphs.\n\n"
            "Convert the SPARQL query into a clear natural language question.\n\n"
            f"### SPARQL\n{sparql}\n\n"
            "### Question\n"
        )

    @torch.no_grad()
    def generate_texts(
        self,
        sparql_list: List[str],
        batch_size: int = 1,
        max_new_tokens: int = 32,
    ) -> List[str]:
        generated = []

        model_device = next(self.qwen_model.parameters()).device

        for start in tqdm(range(0, len(sparql_list), batch_size), desc="Qwen generate"):
            batch = sparql_list[start : start + batch_size]
            prompts = [self.build_prompt(q) for q in batch]

            toks = self.qwen_tokenizer(
                prompts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1024,
            )
            toks = {k: v.to(model_device) for k, v in toks.items()}

            out_ids = self.qwen_model.generate(
                **toks,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=1,
                pad_token_id=self.qwen_tokenizer.pad_token_id,
                eos_token_id=self.qwen_tokenizer.eos_token_id,
                use_cache=True,
            )

            input_lengths = toks["attention_mask"].sum(dim=1).tolist()

            for i in range(len(batch)):
                gen_ids = out_ids[i][input_lengths[i]:]
                text = self.qwen_tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
                generated.append(text)

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

        return generated

    def encode_anchor_sparql(self, sparql_list: List[str], gen_batch_size: int = 1, max_new_tokens: int = 32):
        generated_texts = self.generate_texts(
            sparql_list,
            batch_size=gen_batch_size,
            max_new_tokens=max_new_tokens,
        )
        embeddings = self.sentence_encoder.encode(
            generated_texts,
            batch_size=args.sentence_batch_size,
        )
        return embeddings, generated_texts

    def encode_corpus_questions(self, question_list: List[str]) -> np.ndarray:
        return self.sentence_encoder.encode(
            question_list,
            batch_size=args.sentence_batch_size,
        )


# ---------------------------
# Metrics
# ---------------------------
def mask_self_matches(sim_matrix: np.ndarray) -> np.ndarray:
    sim = sim_matrix.copy()
    n, m = sim.shape
    if n == m:
        diag = np.arange(n)
        sim[diag, diag] = -1e9
    return sim


def get_ranked_indices(sim_matrix: np.ndarray) -> np.ndarray:
    return np.argsort(-sim_matrix, axis=1)


def recall_at_k(ranked_indices: np.ndarray, gt: Dict[int, List[int]], ks: List[int]) -> Dict[str, float]:
    ks = sorted(ks)
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]

    scores = {}
    for k in ks:
        hits = 0
        for i in valid_anchors:
            gt_set = set(gt[i])
            topk = ranked_indices[i, :k]
            if len(gt_set.intersection(topk)) > 0:
                hits += 1
        scores[f"Recall@{k}"] = hits / max(len(valid_anchors), 1)
    return scores


def mrr_at_k(ranked_indices: np.ndarray, gt: Dict[int, List[int]], k: int) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    rr = []

    for i in valid_anchors:
        gt_set = set(gt[i])
        topk = ranked_indices[i, :k]

        reciprocal_rank = 0.0
        for rank, idx in enumerate(topk, start=1):
            if idx in gt_set:
                reciprocal_rank = 1.0 / rank
                break
        rr.append(reciprocal_rank)

    return float(np.mean(rr)) if rr else 0.0


def dcg_at_k(relevances: List[float], k: int) -> float:
    rel = np.asarray(relevances[:k], dtype=np.float32)
    if rel.size == 0:
        return 0.0
    discounts = 1.0 / np.log2(np.arange(2, rel.size + 2))
    return float(np.sum(rel * discounts))


def ndcg_at_k(ranked_indices: np.ndarray, relevance_matrix: np.ndarray, gt: Dict[int, List[int]], k: int) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    vals = []

    for i in valid_anchors:
        topk = ranked_indices[i, :k]
        rels = [float(relevance_matrix[i, j]) for j in topk]
        dcg = dcg_at_k(rels, k)

        ideal_rels = sorted(
            [float(relevance_matrix[i, j]) for j in range(relevance_matrix.shape[1]) if j != i],
            reverse=True,
        )
        idcg = dcg_at_k(ideal_rels, k)
        vals.append(0.0 if idcg == 0.0 else dcg / idcg)

    return float(np.mean(vals)) if vals else 0.0


def best_jaccard_at_k(ranked_indices: np.ndarray, jaccard_matrix: np.ndarray, gt: Dict[int, List[int]], k: int) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    vals = []

    for i in valid_anchors:
        topk = ranked_indices[i, :k]
        vals.append(float(np.max(jaccard_matrix[i, topk])) if len(topk) > 0 else 0.0)

    return float(np.mean(vals)) if vals else 0.0


def mean_jaccard_at_k(ranked_indices: np.ndarray, jaccard_matrix: np.ndarray, gt: Dict[int, List[int]], k: int) -> float:
    valid_anchors = [i for i in range(ranked_indices.shape[0]) if len(gt.get(i, [])) > 0]
    vals = []

    for i in valid_anchors:
        topk = ranked_indices[i, :k]
        vals.append(float(np.mean(jaccard_matrix[i, topk])) if len(topk) > 0 else 0.0)

    return float(np.mean(vals)) if vals else 0.0


def evaluate_ranking(sim_matrix: np.ndarray, gt: Dict[int, List[int]], jaccard_matrix: np.ndarray, ks: List[int]) -> Dict[str, float]:
    sim_matrix = mask_self_matches(sim_matrix)
    ranked = get_ranked_indices(sim_matrix)

    results = {}
    results.update(recall_at_k(ranked, gt, ks))

    max_k = max(ks)
    results[f"MRR@{max_k}"] = mrr_at_k(ranked, gt, max_k)

    for k in ks:
        results[f"nDCG@{k}"] = ndcg_at_k(ranked, jaccard_matrix, gt, k)
        results[f"BestJaccard@{k}"] = best_jaccard_at_k(ranked, jaccard_matrix, gt, k)
        results[f"MeanJaccard@{k}"] = mean_jaccard_at_k(ranked, jaccard_matrix, gt, k)

    valid_anchors = sum(1 for i in range(sim_matrix.shape[0]) if len(gt.get(i, [])) > 0)
    results["NumAnchorsWithPositives"] = int(valid_anchors)
    return results


def print_results(title: str, results: Dict[str, float]):
    print(f"\n=== {title} ===")
    for k, v in results.items():
        if k == "NumAnchorsWithPositives":
            print(f"{k}: {v}")
        else:
            print(f"{k}: {v:.4f}")


# ---------------------------
# Retrieval preview
# ---------------------------
def get_topk_indices(sim_matrix: np.ndarray, k: int) -> np.ndarray:
    sim = mask_self_matches(sim_matrix)
    return np.argsort(-sim, axis=1)[:, :k]


def print_retrieval_examples(
    title: str,
    sim_matrix: np.ndarray,
    examples: List[Dict],
    corpus_questions: List[str],
    jaccard_matrix: np.ndarray,
    num_examples: int = 5,
):
    ranked_top5 = get_topk_indices(sim_matrix, k=5)

    print(f"\n===== Retrieval preview: {title} =====")

    shown = 0
    for i in range(len(examples)):
        top5 = ranked_top5[i]
        if len(top5) == 0:
            continue

        print("\n" + "=" * 80)
        print(f"ANCHOR #{i} | id={examples[i]['id']}")
        print(f"SPARQL:\n{examples[i]['sparql']}")
        print(f"GOLD QUESTION:\n{examples[i]['question']}")
        print(f"GOLD ANSWER: {examples[i]['answer']}")

        print("\nTop-5 retrieved:")
        for rank, j in enumerate(top5, start=1):
            print(f"\n[Rank {rank}] corpus_idx={j}")
            print(f"Question: {corpus_questions[j]}")
            print(f"SPARQL: {examples[j]['sparql']}")
            print(f"Answer: {examples[j]['answer']}")
            print(f"Answer-Jaccard(anchor, retrieved) = {jaccard_matrix[i, j]:.4f}")

        print("\nSummary:")
        print(f"Top-1 Jaccard: {jaccard_matrix[i, top5[0]]:.4f}")
        print(f"Best Jaccard in Top-3: {np.max(jaccard_matrix[i, top5[:3]]):.4f}")
        print(f"Best Jaccard in Top-5: {np.max(jaccard_matrix[i, top5[:5]]):.4f}")

        shown += 1
        if shown >= num_examples:
            break


# ---------------------------
# Main notebook runner
# ---------------------------
def run_eval(args):
    cleanup_memory()
    set_seed(args.seed)
    os.makedirs(args.save_dir, exist_ok=True)

    print(f"Using device: {DEVICE}")
    print(f"Loading examples from {args.data_path}")

    examples = load_examples(args.data_path, args.use_field, max_examples=args.max_examples)
    print(f"Loaded {len(examples)} usable examples")

    corpus_sparql, anchor_sparql, corpus_questions, gt, jaccard_matrix = build_ground_truth_by_answer_jaccard(
        examples,
        threshold=args.answer_jaccard_threshold,
        exclude_self=True,
    )

    num_with_pos = sum(1 for i in range(len(anchor_sparql)) if len(gt[i]) > 0)
    print(f"Anchors with at least one positive: {num_with_pos}/{len(anchor_sparql)}")

    all_results = {
        "config": args.__dict__.copy(),
        "dataset_stats": {
            "num_examples": len(examples),
            "num_anchors_with_positives": num_with_pos,
        },
        "results": {},
    }

    if args.run_sparqlce:
        print("\nEncoding corpus with SPARQLCE...")
        corpus_emb_sparqlce = encode_sparqlce_batch(corpus_sparql)

        print("Encoding anchors with SPARQLCE...")
        anchor_emb_sparqlce = encode_sparqlce_batch(anchor_sparql)

        sim_sparqlce = cosine_sim(anchor_emb_sparqlce, corpus_emb_sparqlce)

        sparqlce_results = evaluate_ranking(
            sim_matrix=sim_sparqlce,
            gt=gt,
            jaccard_matrix=jaccard_matrix,
            ks=args.topk,
        )
        print_results("SPARQLCE", sparqlce_results)

        print_retrieval_examples(
            title="SPARQLCE",
            sim_matrix=sim_sparqlce,
            examples=examples,
            corpus_questions=corpus_questions,
            jaccard_matrix=jaccard_matrix,
            num_examples=args.preview_examples,
        )

        all_results["results"]["sparqlce"] = sparqlce_results

    if args.run_qwen:
        print("\nLoading Qwen baseline...")
        qwen = QwenTextBaseline(
            qwen_path=args.qwen_path,
            sentence_model_name=args.sentence_model,
            device=DEVICE,
        )

        print("Encoding corpus gold questions...")
        corpus_emb_qwen = qwen.encode_corpus_questions(corpus_questions)

        print("Encoding anchor SPARQL via Qwen...")
        anchor_emb_qwen, generated_anchor_texts = qwen.encode_anchor_sparql(
            anchor_sparql,
            gen_batch_size=args.qwen_gen_batch_size,
            max_new_tokens=args.qwen_max_new_tokens,
        )

        sim_qwen = cosine_sim(anchor_emb_qwen, corpus_emb_qwen)

        qwen_results = evaluate_ranking(
            sim_matrix=sim_qwen,
            gt=gt,
            jaccard_matrix=jaccard_matrix,
            ks=args.topk,
        )
        print_results("Qwen SPARQL->Text baseline", qwen_results)

        print_retrieval_examples(
            title="Qwen SPARQL->Text baseline",
            sim_matrix=sim_qwen,
            examples=examples,
            corpus_questions=corpus_questions,
            jaccard_matrix=jaccard_matrix,
            num_examples=args.preview_examples,
        )

        all_results["results"]["qwen_sparql2text"] = qwen_results

        preview = []
        for i in range(min(50, len(anchor_sparql))):
            preview.append(
                {
                    "id": examples[i]["id"],
                    "sparql": anchor_sparql[i],
                    "gold_question": corpus_questions[i],
                    "generated_question": generated_anchor_texts[i],
                }
            )

        preview_path = os.path.join(args.save_dir, "qwen_generated_anchor_text_preview.json")
        save_json(preview, preview_path)
        print(f"Saved Qwen preview to {preview_path}")

    results_path = os.path.join(args.save_dir, "rq1_retrieval_results.json")
    save_json(all_results, results_path)
    print(f"\nSaved results to {results_path}")

    return all_results

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql-emb/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
results = run_eval(args)

Using device: cuda
Loading examples from dataset/test.json
Loaded 3052 usable examples


Building answer-Jaccard matrix: 100%|██████████| 3052/3052 [02:03<00:00, 24.78it/s]


Anchors with at least one positive: 562/3052

Loading Qwen baseline...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 391.63it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Encoding corpus gold questions...


Encoding anchor SPARQL via Qwen...


Qwen generate: 100%|██████████| 3052/3052 [1:27:15<00:00,  1.72s/it]



=== Qwen SPARQL->Text baseline ===
Recall@1: 0.0676
Recall@5: 0.1406
Recall@10: 0.2171
MRR@10: 0.1020
nDCG@1: 0.0723
BestJaccard@1: 0.0722
MeanJaccard@1: 0.0722
nDCG@5: 0.0640
BestJaccard@5: 0.1501
MeanJaccard@5: 0.0364
nDCG@10: 0.0748
BestJaccard@10: 0.2271
MeanJaccard@10: 0.0323
NumAnchorsWithPositives: 562

===== Retrieval preview: Qwen SPARQL->Text baseline =====

ANCHOR #0 | id=0
SPARQL:
SELECT ?answer WHERE { ?statement1 <http://www.w3.org/1999/02/22-rdf-syntax-ns#subject> <http://wikidata.dbpedia.org/resource/Q16538> . ?statement1 <http://www.w3.org/1999/02/22-rdf-syntax-ns#predicate> <http://www.wikidata.org/entity/P725>. ?statement1 <http://www.w3.org/1999/02/22-rdf-syntax-ns#object> ?answer . ?statement2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#subject> ?answer. ?statement2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#predicate> <http://www.wikidata.org/entity/P106> . ?statement2 <http://www.w3.org/1999/02/22-rdf-syntax-ns#object> <http://wikidata.dbpedia.org/resource/Q177

In [3]:
print(args.use_field)

sparql_dbpedia18
